In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style global
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_palette('husl')

AIRPORTS = ['Ajaccio', 'Bastia', 'Bron', 'Nantes', 'Biarritz', 'Pise']
AIRPORT_COORDS = {
    'Bron':    (4.9389,  45.7294),
    'Bastia':  (9.4837,  42.5527),
    'Ajaccio': (8.8029,  41.9236),
    'Nantes':  (-1.6107, 47.1532),
    'Pise':    (10.399,  43.695),
    'Biarritz':(-1.524,  43.4683),
}
print('✅ Imports OK')

✅ Imports OK


In [3]:
path = '..\\data\\meteo_data_clipped.pkl'
meteo_data = pd.read_pickle(path)


UnpicklingError: invalid load key, '\x07'.

In [ ]:
def build_sequences(df_all, instant_features, target_col, seq_len,
                    window_min=30):
    """
    Construit les séquences d'entrée pour le LSTM.

    Principe
    --------
    Pour chaque éclair CG ≤ 20km ayant une cible renseignée, on
    remonte les `seq_len` éclairs les plus récents (CG + IC) du même
    aéroport survenus dans les `window_min` minutes précédentes.

    Chaque pas de temps de la séquence est décrit par ses features
    INSTANTANÉES uniquement (delta_t, amplitude, dist, azimuth,
    icloud, maxis). Les agrégats temporels (n_cg_10min, ratio_ic_cg,
    etc.) sont exclus car le LSTM les apprend implicitement depuis
    la séquence brute.

    Paramètres
    ----------
    df_all           : DataFrame — tous les éclairs (CG + IC), triés
                       par ['airport', 'date'], avec les features
                       instantanées et la colonne target_col
    instant_features : list[str] — features instantanées par éclair
                       (delta_t_sec, amplitude, dist, azimuth,
                        icloud, maxis, ...)
    target_col       : str — temps en secondes jusqu'au prochain
                       éclair CG ≤ 20km (NaN pour les non-cibles)
    seq_len          : int — nombre max d'éclairs dans le contexte
                       (tronqué si plus de seq_len dans la fenêtre)
    window_min       : int — durée max du contexte en minutes
                       (défaut 30 min)

    Structure d'une séquence
    ------------------------
    Éclair CG cible à t0 :
        fenêtre   = ]t0 - window_min, t0[   — strictement avant t0
        contexte  = éclairs (CG+IC) dans la fenêtre
                    → tronqué aux seq_len plus récents si besoin
                    → paddé à gauche par des zéros si moins de seq_len

    Masque
    ------
        mask[j] = 1  si pas j est un éclair réel
        mask[j] = 0  si pas j est du padding

    Returns
    -------
    X    : np.ndarray (N, seq_len, n_features) — séquences
    mask : np.ndarray (N, seq_len)             — 1=réel, 0=padding
    y    : np.ndarray (N,)                     — cible (secondes)
    idx  : np.ndarray (N,)                     — index dans df_all

    Exemple
    -------
    >>> INSTANT_FEATURES = ['delta_t_sec', 'amplitude', 'dist',
    ...                     'azimuth', 'icloud', 'maxis']
    >>> X, mask, y, idx = build_sequences(
    ...     df_all           = df,
    ...     instant_features = INSTANT_FEATURES,
    ...     target_col       = 'target',
    ...     seq_len          = 10,
    ...     window_min       = 30
    ... )
    >>> X.shape
    (84302, 10, 6)
    """

    df_all   = df_all.sort_values(['airport', 'date']).reset_index(drop=True)
    n_feat   = len(instant_features)
    features = df_all[instant_features].values.astype(np.float32)
    targets  = df_all[target_col].values
    dates    = df_all['date'].values

    # Observations cibles : CG ≤ 20km avec target renseignée
    is_target = (
        (df_all['icloud'] == False) &
        (df_all['dist']   <= 20)    &
        (df_all[target_col].notna())
    ).values

    window_ns = np.timedelta64(window_min, 'm')

    X_list, mask_list, y_list, idx_list = [], [], [], []

    for _, grp in df_all.groupby('airport'):

        positions  = grp.index.tolist()
        grp_dates  = dates[positions]
        is_tgt_grp = is_target[positions]

        for pos, abs_idx in enumerate(positions):

            if not is_tgt_grp[pos]:
                continue

            t0 = grp_dates[pos]

            # Fenêtre temporelle : ]t0 - 30min, t0[
            t_min = t0 - window_ns
            ctx   = [
                positions[j]
                for j in range(pos)                      # strictement avant
                if grp_dates[j] > t_min                  # dans la fenêtre
            ]

            # Tronquer aux seq_len plus récents
            if len(ctx) > seq_len:
                ctx = ctx[-seq_len:]

            n_real  = len(ctx)
            pad_len = seq_len - n_real

            seq_real = features[ctx] if n_real > 0 \
                       else np.empty((0, n_feat), dtype=np.float32)

            if pad_len > 0:
                pad     = np.zeros((pad_len, n_feat), dtype=np.float32)
                seq_pad = np.vstack([pad, seq_real]) if n_real > 0 else pad
            else:
                seq_pad = seq_real

            mask = np.array([0]*pad_len + [1]*n_real, dtype=np.float32)

            X_list.append(seq_pad)
            mask_list.append(mask)
            y_list.append(float(targets[abs_idx]))
            idx_list.append(abs_idx)

    return (
        np.array(X_list,    dtype=np.float32),
        np.array(mask_list, dtype=np.float32),
        np.array(y_list,    dtype=np.float32),
        np.array(idx_list,  dtype=np.int64)
    )


# ── Utilisation ───────────────────────────────────────────────────────────

# Features instantanées UNIQUEMENT — pas d'agrégats temporels
INSTANT_FEATURES = [
    'delta_t_sec',   # intervalle depuis l'éclair précédent
    'amplitude',     # intensité et polarité
    'dist',          # distance à l'aéroport
    'azimuth',       # direction
    'icloud',        # type : 0=CG, 1=IC
    'maxis',         # erreur de localisation
]

X, mask, y, idx = build_sequences(
    df_all           = df,
    instant_features = INSTANT_FEATURES,
    target_col       = 'target',
    seq_len          = 10,
    window_min       = 30
)

print(f"X    : {X.shape}      → (N, seq_len={10}, n_features={len(INSTANT_FEATURES)})")
print(f"mask : {mask.shape}")
print(f"y    : {y.shape}")

# Contrôle rapide
n_real_per_seq = mask.sum(axis=1)
print(f"\nÉclairs réels par séquence :")
for k in range(0, 11):
    pct = (n_real_per_seq == k).mean() * 100
    if pct > 0.1:
        bar = '█' * int(pct / 2)
        print(f"  {k:>2} éclairs : {pct:5.1f}%  {bar}")